# Финальный большой прогон: USER-bge-m3 @ seq320 (самодостаточный)

Рецепт собран из проверенного командой: stage-A only (2 эпохи, soft labels) —
A/B на LB; swap-аугментация — +0.014 на абляции; seq 320 — контекст дал +0.011
уже при 224, а в 160 влезает лишь треть пар; confidence weighting — рецепт Мишани;
best-checkpoint по сбалансированной fast-валидации.

**Нужно 2 файла**: items.parquet, matches_llm.parquet (пути в конфиге).
Больше ничего: ни репозитория, ни полигона.

**Время**: ~14-20ч на A100, ~7-10ч на H100 (2 эпохи по 11M пар).
Если надо быстрее: EPOCHS=1 (вдвое; вторая эпоха давала +0.026 locally — жалко),
или MODEL_NAME='intfloat/multilingual-e5-base' (~4x быстрее, ожидаемо -0.01 LB).

**Чекпойнты**: best (по fast-val) и last сохраняются в OUT_DIR каждые EVAL_EVERY
шагов — обрыв не теряет прогресс. Итог: папка экспорта модели + metrics.json.
Что прислать в чат: metrics.json + путь/архив папки export.


In [ ]:
# ===== КОНФИГ =====
ITEMS_PATH   = "./items.parquet"
MATCHES_PATH = "./matches_llm.parquet"
OUT_DIR      = "./bge_m3_320_run"     # чекпойнты, экспорт, метрики

MODEL_NAME = "deepvk/USER-bge-m3"
MAX_LEN    = 320
EPOCHS     = 2
LR         = 2e-5
WARMUP_FRAC = 0.05
SWAP_P     = 0.5
EVAL_EVERY = 8000        # опт-шагов между fast-val/чекпойнтами

In [ ]:
import os, json, gc, time, random
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda"
os.makedirs(OUT_DIR, exist_ok=True)
CC = torch.cuda.get_device_capability(0)
AMP = torch.bfloat16 if CC[0] >= 8 else torch.float16
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 128 if gpu_gb > 70 else (64 if gpu_gb > 45 else 32)
ACCUM = max(1, 256 // BATCH)
print(f"{torch.cuda.get_device_name(0)} {gpu_gb:.0f}GB | amp={AMP} batch={BATCH} accum={ACCUM}")

In [ ]:
# данные: групповой сплит seed 13 (общекомандный), тексты v1
ml = pd.read_parquet(MATCHES_PATH)
parent = {}
def find(x):
    p = parent.setdefault(x, x)
    while p != parent[p]:
        parent[p] = parent[parent[p]]; p = parent[p]
    parent[p] = p; return p
for a, b in zip(ml.id1.values, ml.id2.values):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
comp = np.fromiter((find(i) for i in ml.id1.values), dtype=np.int64, count=len(ml))
rng = np.random.RandomState(13)
u = np.unique(comp)
vs = set(u[rng.rand(len(u)) < 0.03].tolist())
is_val = np.fromiter((c in vs for c in comp), dtype=bool, count=len(ml))
train = ml[~is_val].reset_index(drop=True)
holdout = ml[is_val].copy()
holdout = holdout[(holdout.target <= 0.2) | (holdout.target >= 0.8)]
holdout["target"] = (holdout.target >= 0.5).astype(np.int8)
del ml, parent, comp; gc.collect()
print(f"train {len(train):,} | holdout {len(holdout):,} (ожидание 10,950,394 / 191,555)")

KEYS = ["бренд", "артикул", "партномер", "oem", "код", "модель", "размер",
        "цвет", "объем", "обьем", "вес", "тип", "материал", "количество"]
def build_text(name, attributes):
    parts = [str(name) if name is not None else ""]
    try: attrs = json.loads(attributes) if isinstance(attributes, str) else {}
    except Exception: attrs = {}
    if isinstance(attrs, dict) and attrs:
        low = {str(k).lower(): str(v) for k, v in attrs.items() if v}
        picked, used = [], set()
        for w in KEYS:
            for k, v in low.items():
                if w in k and k not in used:
                    picked.append(f"{k}:{v}"); used.add(k)
        rest = [f"{k}:{v}" for k, v in low.items() if k not in used]
        parts.append(" ; ".join(picked + rest)[:520])   # 520 симв под seq 320
    return " | ".join(parts)

texts, cats = {}, {}
f = pq.ParquetFile(ITEMS_PATH)
for b in f.iter_batches(columns=["id", "name", "attributes", "category"], batch_size=500_000):
    df = b.to_pandas()
    for i, n, a, c in df.itertuples(index=False, name=None):
        texts[i] = build_text(n, a); cats[i] = c
holdout["category"] = [cats[i] for i in holdout.id1]

# сбалансированная fast-валидация: до 3000 пар на категорию
hf = holdout.sample(frac=1, random_state=0).groupby("category").head(3000)
print(f"товаров: {len(texts):,}; fast-val: {len(hf):,}")

In [ ]:
class DS(Dataset):
    def __init__(self, df, training=False):
        self.a, self.b = df.id1.values, df.id2.values
        self.y = df.target.values.astype(np.float32)
        self.training = training
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        a, b = self.a[i], self.b[i]
        if self.training and random.random() < SWAP_P:
            a, b = b, a
        return texts[a], texts[b], self.y[i]

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)

def collate(batch):
    enc = tok([x[0] for x in batch], [x[1] for x in batch], padding=True,
              truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return enc, torch.tensor([x[2] for x in batch])

@torch.no_grad()
def macro_on(df, bs=256):
    model.eval(); preds = []
    dl = DataLoader(DS(df), batch_size=bs, num_workers=0, shuffle=False,
                    collate_fn=lambda b: tok([x[0] for x in b], [x[1] for x in b],
                        padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"))
    for enc in dl:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", AMP):
            preds.append(torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy())
    z = df[["category", "target"]].copy(); z["p"] = np.concatenate(preds)
    model.train()
    return float(z.groupby("category").apply(lambda g: average_precision_score(g.target, g.p)).mean())

dl = DataLoader(DS(train, training=True), batch_size=BATCH, shuffle=True,
                num_workers=0, drop_last=True, collate_fn=collate)
steps_total = len(dl) * EPOCHS // ACCUM
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, int(steps_total * WARMUP_FRAC), steps_total)
scaler = torch.amp.GradScaler(enabled=AMP == torch.float16)

best = 0.0
step = 0
t0 = time.time()
model.train()
for ep in range(EPOCHS):
    for bi, (enc, y) in enumerate(dl):
        enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
        y = y.to(device, non_blocking=True)
        with torch.autocast("cuda", AMP):
            per = F.binary_cross_entropy_with_logits(
                model(**enc).logits.squeeze(-1), y, reduction="none")
            w = 0.75 + 0.25 * (2 * y - 1).abs()          # confidence weighting
            loss = (per * w).sum() / w.sum().clamp_min(1e-6) / ACCUM
        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()
        if (bi + 1) % ACCUM == 0:
            if scaler.is_enabled():
                scaler.step(opt); scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True); sched.step(); step += 1
            if step % 1000 == 0:
                sps = step * BATCH * ACCUM / (time.time() - t0)
                print(f"ep{ep} step {step}/{steps_total} loss={loss.item()*ACCUM:.4f} {sps:.0f} pair/s", flush=True)
            if step % EVAL_EVERY == 0:
                mac = macro_on(hf)
                print(f"  >> step {step}: fast-val macro = {mac:.4f} (best {best:.4f})", flush=True)
                torch.save(model.state_dict(), f"{OUT_DIR}/last.pt")
                if mac > best:
                    best = mac
                    torch.save(model.state_dict(), f"{OUT_DIR}/best.pt")
print("обучение завершено; best fast-val:", best)

In [ ]:
# финал: лучший чекпойнт -> полный holdout -> экспорт
sd = torch.load(f"{OUT_DIR}/best.pt", map_location="cpu", weights_only=True)
model.load_state_dict(sd)
full = macro_on(holdout, bs=256)
print(f"ПОЛНЫЙ llm-holdout macro PR-AUC = {full:.4f}")
print("ориентиры команды: tiny-2ep 0.771, e5-small 0.786; калибровка LB ~= 0.6 x holdout")

model.save_pretrained(f"{OUT_DIR}/export")
tok.save_pretrained(f"{OUT_DIR}/export")
json.dump({"model": MODEL_NAME, "max_len": MAX_LEN, "epochs": EPOCHS, "swap": SWAP_P,
           "confidence_weighting": True, "full_llm_holdout_macro": full,
           "best_fastval": best}, open(f"{OUT_DIR}/metrics.json", "w"), indent=1)
print("экспорт:", f"{OUT_DIR}/export", "| metrics.json готов")